# Cultural Artifact — Interactive 3D Viewer (Output Notebook)

**Purpose:** Build the main interactive viewer required by the project plan, using the already-exported Nerfacto and Splatfacto PLY files. This notebook does **not** train either model and does **not** modify the original checkpoints.

## What this notebook creates
- `index.html`: one viewer page with a Splatfacto Gaussian-splat view and a Nerfacto point-cloud view.
- `Artifact_Interactive_3D_Viewer.zip`: HTML + both PLY files, ready to download and run locally.
- Mouse orbit/rotation, scroll zoom, pan, reset, and front/top/side camera presets.

**Important:** Splatfacto is rendered as Gaussian splats using Spark. Nerfacto's current export is a point cloud, so that tab is a point-cloud view, not a solid mesh. The PLY files do not contain the original camera poses; front/top/side are convenient camera presets, while arbitrary orbiting is limited by the reconstructed geometry and training-image coverage.

### Before running
In Kaggle, attach a private dataset containing these two files:
- `ceramic_splatfacto.ply`
- `point_cloud.ply`

You can create that dataset by uploading the two PLY files you already downloaded. Then attach it to this notebook using **Add Input**. Do not attach or rerun the training notebooks.

## 1. Check GPU/browser-independent environment and locate the exported models
This cell searches `/kaggle/input` for the two PLY files. It only reads the files; it does not retrain or re-export models.

In [2]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
splat_matches = list(INPUT_ROOT.rglob("ceramic_splatfacto.ply"))
nerf_matches = list(INPUT_ROOT.rglob("point_cloud.ply"))

print("Splatfacto PLY matches:")
for p in splat_matches:
    print(" ", p, f"({p.stat().st_size / 1024**2:.2f} MB)")

print("\nNerfacto PLY matches:")
for p in nerf_matches:
    print(" ", p, f"({p.stat().st_size / 1024**2:.2f} MB)")

if not splat_matches or not nerf_matches:
    raise FileNotFoundError(
        "Both PLY files must be in an attached Kaggle input dataset. "
        "Upload ceramic_splatfacto.ply and point_cloud.ply to a private Kaggle dataset, "
        "then attach it using Add Input."
    )

SPLAT_SOURCE = splat_matches[0]
NERF_SOURCE = nerf_matches[0]
print("\nSelected files:")
print("Splatfacto:", SPLAT_SOURCE)
print("Nerfacto :", NERF_SOURCE)


Splatfacto PLY matches:
  /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/ceramic_splatfacto.ply (24.23 MB)

Nerfacto PLY matches:
  /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/point_cloud.ply (12.92 MB)

Selected files:
Splatfacto: /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/ceramic_splatfacto.ply
Nerfacto : /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/point_cloud.ply


## 2. Copy the two existing model exports into the viewer package
The original input files remain unchanged.

In [3]:
from pathlib import Path
import shutil

OUTPUT_DIR = Path("/kaggle/working/Artifact_Interactive_3D_Viewer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(SPLAT_SOURCE, OUTPUT_DIR / "ceramic_splatfacto.ply")
shutil.copy2(NERF_SOURCE, OUTPUT_DIR / "point_cloud.ply")

for name in ["ceramic_splatfacto.ply", "point_cloud.ply"]:
    p = OUTPUT_DIR / name
    print(f"{name}: {p.stat().st_size / 1024**2:.2f} MB")


ceramic_splatfacto.ply: 24.23 MB
point_cloud.ply: 12.92 MB


## 3. Create the interactive viewer page

The Splatfacto tab uses Spark's `SplatMesh` and `SparkRenderer` to render Gaussian primitives. The Nerfacto tab uses Three.js `PLYLoader` to display the exported coloured point cloud. Both tabs provide orbit, zoom, pan and camera presets.

The page loads JavaScript libraries from their public CDNs, so an internet connection is required when opening the viewer.

In [4]:
from pathlib import Path

html = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Vietnamese Ceramic Artifact — Interactive 3D Viewer</title>
<style>
:root { color-scheme: dark; }
* { box-sizing: border-box; }
body { margin:0; background:#0c1422; color:#eef4ff; font-family:Arial,sans-serif; }
header { padding:18px 24px; background:#17243a; }
h1 { margin:0 0 7px; font-size:23px; }
header p { margin:0; color:#c4d0e0; }
.toolbar { display:flex; flex-wrap:wrap; align-items:center; gap:9px; padding:13px 18px; background:#111d30; }
button { padding:9px 14px; border:1px solid #53657d; border-radius:6px; background:#243650; color:white; cursor:pointer; }
button.active { background:#1767a8; border-color:#69b9ff; }
button:hover { background:#315071; }
#viewer { width:100%; height:72vh; min-height:480px; position:relative; }
#viewer canvas { display:block; width:100%; height:100%; }
#status { padding:10px 18px; color:#9bd5ff; background:#111d30; min-height:40px; }
.hint { padding:0 18px 14px; color:#bdc9d9; font-size:13px; background:#111d30; }
.hidden { display:none !important; }
.spacer { flex:1; }
@media(max-width:600px){ #viewer{height:65vh;min-height:380px;} header{padding:15px;} }
</style>
<script type="importmap">
{
 "imports": {
  "three": "https://cdn.jsdelivr.net/npm/three@0.180.0/build/three.module.js",
  "three/addons/": "https://cdn.jsdelivr.net/npm/three@0.180.0/examples/jsm/",
  "@sparkjsdev/spark": "https://sparkjs.dev/releases/spark/2.4.0/spark.module.js"
 }
}
</script>
</head>
<body>
<header>
<h1>Vietnamese Ceramic Artifact — Interactive 3D Viewer</h1>
<p>Digital preservation using Nerfacto and 3D Gaussian Splatting</p>
</header>
<div class="toolbar">
 <button id="splatTab" class="active">Splatfacto — Gaussian Splats</button>
 <button id="nerfTab">Nerfacto — Point Cloud</button>
 <span class="spacer"></span>
 <button id="reset">Reset view</button>
 <button id="front">Front</button>
 <button id="top">Top</button>
 <button id="side">Side</button>
</div>
<div id="viewer"></div>
<div id="status">Loading Splatfacto Gaussian model…</div>
<div class="hint">Controls: left-drag = orbit/rotate · mouse wheel = zoom · right-drag = pan. Use the tabs to switch reconstruction methods.</div>

<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";
import { PLYLoader } from "three/addons/loaders/PLYLoader.js";
import { SparkRenderer, SplatMesh } from "@sparkjsdev/spark";

const host = document.getElementById("viewer");
const status = document.getElementById("status");
const scene = new THREE.Scene();
scene.background = new THREE.Color(0x0c1422);
const camera = new THREE.PerspectiveCamera(50, host.clientWidth / host.clientHeight, 0.001, 10000);
const renderer = new THREE.WebGLRenderer({ antialias: false, alpha: false });
renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
renderer.setSize(host.clientWidth, host.clientHeight);
host.appendChild(renderer.domElement);

const sparkRenderer = new SparkRenderer({ renderer });
scene.add(sparkRenderer);

const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;
controls.dampingFactor = 0.07;
controls.screenSpacePanning = true;

let splatObject = null;
let pointObject = null;
let currentMode = "splat";
let fitRadius = 1;

function setCamera(which) {
  const r = fitRadius;
  if (which === "top") camera.position.set(0, r * 3.0, 0.001);
  else if (which === "side") camera.position.set(r * 3.0, 0, 0);
  else camera.position.set(0, 0, r * 3.0);
  controls.target.set(0, 0, 0);
  camera.near = Math.max(r / 1000, 0.001);
  camera.far = Math.max(r * 100, 100);
  camera.updateProjectionMatrix();
  controls.update();
}
function setMode(mode) {
  currentMode = mode;
  if (splatObject) splatObject.visible = mode === "splat";
  if (pointObject) pointObject.visible = mode === "nerf";
  document.getElementById("splatTab").classList.toggle("active", mode === "splat");
  document.getElementById("nerfTab").classList.toggle("active", mode === "nerf");
  status.textContent = mode === "splat"
    ? "Splatfacto Gaussian-splat reconstruction"
    : "Nerfacto exported coloured point-cloud reconstruction";
}
function fitFromBox(box) {
  const center = box.getCenter(new THREE.Vector3());
  const size = box.getSize(new THREE.Vector3());
  const maxDim = Math.max(size.x, size.y, size.z);
  fitRadius = Math.max(maxDim * 0.5, 0.01);
  controls.target.copy(center);
  setCamera("front");
}
async function loadSplat() {
  status.textContent = "Loading Gaussian splats… this may take a little time.";
  try {
    splatObject = new SplatMesh({ url: "./ceramic_splatfacto.ply", lod: true });
    scene.add(splatObject);
    await splatObject.initialized;
    const box = splatObject.getBoundingBox(true);
    fitFromBox(box);
    splatObject.visible = true;
    status.textContent = "Splatfacto loaded: Gaussian splats are being rendered.";
  } catch (err) {
    console.error(err);
    status.textContent = "Could not load Splatfacto. Check that ceramic_splatfacto.ply is beside index.html and that the browser has internet access. Details: " + err.message;
  }
}
async function loadNerf() {
  status.textContent = "Loading Nerfacto point cloud…";
  try {
    const geometry = await new PLYLoader().loadAsync("./point_cloud.ply");
    geometry.computeBoundingBox();
    const center = geometry.boundingBox.getCenter(new THREE.Vector3());
    const size = geometry.boundingBox.getSize(new THREE.Vector3());
    geometry.translate(-center.x, -center.y, -center.z);
    fitRadius = Math.max(size.x, size.y, size.z) * 0.5;
    const material = new THREE.PointsMaterial({
      size: Math.max(fitRadius * 0.003, 0.002),
      sizeAttenuation: true,
      vertexColors: Boolean(geometry.getAttribute("color"))
    });
    pointObject = new THREE.Points(geometry, material);
    pointObject.visible = false;
    scene.add(pointObject);
    controls.target.set(0, 0, 0);
    setCamera("front");
    status.textContent = "Nerfacto point cloud loaded. Note: this export is points, not a solid mesh.";
  } catch (err) {
    console.error(err);
    status.textContent = "Could not load Nerfacto point cloud. Check that point_cloud.ply is beside index.html. Details: " + err.message;
  }
}
document.getElementById("splatTab").addEventListener("click", () => setMode("splat"));
document.getElementById("nerfTab").addEventListener("click", () => setMode("nerf"));
document.getElementById("reset").addEventListener("click", () => setCamera("front"));
document.getElementById("front").addEventListener("click", () => setCamera("front"));
document.getElementById("top").addEventListener("click", () => setCamera("top"));
document.getElementById("side").addEventListener("click", () => setCamera("side"));

renderer.setAnimationLoop(() => {
  controls.update();
  renderer.render(scene, camera);
});
window.addEventListener("resize", () => {
  camera.aspect = host.clientWidth / host.clientHeight;
  camera.updateProjectionMatrix();
  renderer.setSize(host.clientWidth, host.clientHeight);
});

await loadSplat();
await loadNerf();
setMode("splat");
</script>
</body>
</html>
"""

html_path = OUTPUT_DIR / "index.html"
html_path.write_text(html, encoding="utf-8")
print("Viewer HTML created:", html_path)
print("Size:", round(html_path.stat().st_size / 1024, 1), "KB")


Viewer HTML created: /kaggle/working/Artifact_Interactive_3D_Viewer/index.html
Size: 7.1 KB


## 4. Package the viewer and both model files
The ZIP is the main downloadable output. Keep the two PLY files in the same folder as `index.html` when extracting.

In [5]:
import zipfile

ZIP_PATH = Path("/kaggle/working/Artifact_Interactive_3D_Viewer.zip")
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in ["index.html", "ceramic_splatfacto.ply", "point_cloud.ply"]:
        zf.write(OUTPUT_DIR / filename, arcname=filename)

print("Created:", ZIP_PATH)
print("ZIP size:", round(ZIP_PATH.stat().st_size / 1024**2, 2), "MB")
print("Included:")
for name in ["index.html", "ceramic_splatfacto.ply", "point_cloud.ply"]:
    print(" -", name)


Created: /kaggle/working/Artifact_Interactive_3D_Viewer.zip
ZIP size: 30.4 MB
Included:
 - index.html
 - ceramic_splatfacto.ply
 - point_cloud.ply


## 5. Final output checks

If all three files are present and the ZIP is created, use the Kaggle Output panel to download `Artifact_Interactive_3D_Viewer.zip`.

### Open it on Windows
1. Extract the ZIP.
2. Open PowerShell inside the extracted folder.
3. Run `py -m http.server 8000`.
4. Open `http://localhost:8000` in Chrome or Edge.
5. Select the Splatfacto tab to see the Gaussian-splat model; select Nerfacto to see the exported point cloud.

The browser needs internet access to load Spark and Three.js from their CDNs. If the browser shows a WebGL or import error, copy the exact error message before changing the files.

### Requirement coverage
- Mouse drag to orbit: provided by OrbitControls.
- Scroll to zoom: provided by OrbitControls.
- Inspect surface details: zoom and orbit the reconstruction.
- Move between viewpoints: free orbit plus front/top/side presets.
- Compare two methods: tabs switch between Splatfacto and Nerfacto.

**Limitations to state honestly in the report:** Nerfacto is currently a point-cloud export, not a polygon mesh. The viewer's front/top/side positions are camera presets, not recovered original photograph camera poses. The reconstruction only represents detail supported by the input photographs and trained model.